# 🧬 Genome annotation with NTv3

Annotate a genome from a FASTA and get a GFF3, using a trained NTv3
annotation head.

> **⚠️ This needs a data-centre GPU.** One 1 Mb window peaks at about
> **25.3 GB** of device memory, so an A100-40GB is the practical minimum and
> a 24 GB card will not run it. It will not run in free Colab. On CPU it
> works but takes roughly 7.5 days for a mammalian genome.
>
> The last section runs a miniature model on CPU so you can see the output
> format without any of that.


## Installation


In [ ]:
# From a checkout of this repository:
# !pip install .

import logging
from pathlib import Path

# run() logs through a NullHandler, as a library should. Turn it on to
# watch progress: a whole genome takes around 45 minutes.
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s")


## Which models exist

Each registered model brings its weights **and the window length it must be
run at**. Annotating at the wrong window does not raise: it produces a
plausible GFF3 that is wrong, which is why the geometry travels with the
weights instead of being retyped at each call site.


In [ ]:
from nucleotide_transformer_v3.gene_annotation.model_registry import (
    format_model_config,
    format_model_config_list,
)

print(format_model_config_list())


In [ ]:
print(format_model_config("mammals"))


## Annotating a genome

`weights_uri` in each `model_cfg` entry points at a directory holding
`hyperparameters.json` and `flax_model.msgpack`. Point it at wherever you
staged the weights for that clade, then:

```python
from nucleotide_transformer_v3.gene_annotation.annotate_genome import run
from nucleotide_transformer_v3.gene_annotation.config import (
    ProductionAnnotationConfig,
)

run(ProductionAnnotationConfig(
    fasta_paths=["/data/cow/genome.fa"],
    model_cfg="mammals",
    output_path="/data/cow.gff3.gz",
))
```

Or from a shell, which is the same engine behind a Tiberius-shaped
interface:

```bash
ntv3-annotate --genome /data/cow/genome.fa --model_cfg mammals \
  --out /data/cow.gff3.gz
```

**Stage a soft-masked FASTA.** These models were trained on repeat-masked
assemblies where lowercase bases carry their own token ids.


## A runnable miniature, on CPU

The cells below build a tiny randomly-initialised model and annotate a short
synthetic contig with it. The genes are meaningless — the point is to show
the pipeline running and the shape of what it writes, on any machine.


In [ ]:
import dataclasses, json, random, tempfile

from flax import nnx, serialization

from nucleotide_transformer_v3 import pretrained
from nucleotide_transformer_v3.tokenizers import get_ntv3_tokenizer
from nucleotide_transformer_v3.with_tracks_heads import (
    DiscreteConditionedNTv3WithTracksHeads,
    DiscreteConditionedNTv3WithTracksHeadsConfig,
)

work = Path(tempfile.mkdtemp())
model_dir = work / "tiny_model"
model_dir.mkdir()

config = DiscreteConditionedNTv3WithTracksHeadsConfig(
    alphabet_size=16, pad_token_id=1, mask_token_id=2, token_embed_dim=8,
    conv_init_embed_dim=16, embed_dim=32, num_downsamples=2, num_layers=2,
    attention_heads=2, num_kv_heads=1, key_size=16, value_size=16,
    conditions_vocab_size=[], conditions_names=[], conditions_dims=[],
    enable_trunk_conditioning=False, bigwigs_per_file_assembly={},
    number_bigwig_tracks=0, bed_elements_names=["plus"], bed_num_classes=15,
    keep_target_center_fraction=1.0,
    condition_bigwig_head=False, condition_annotation_head=False,
)
model = DiscreteConditionedNTv3WithTracksHeads(config=config, rngs=nnx.Rngs(0))
model.eval()

# Params AND batch statistics: RMSBatchNorm.var_ema is a BatchStat that
# eval-mode normalisation divides by, and a Param-only filter drops it.
(model_dir / pretrained.CONFIG_FILE).write_text(json.dumps(dataclasses.asdict(config)))
(model_dir / pretrained.WEIGHTS_FILE).write_bytes(
    serialization.to_bytes(nnx.state(model, pretrained.VARIABLES).to_pure_dict())
)
(model_dir / pretrained.TOKENIZER_FILE).write_text(
    json.dumps(get_ntv3_tokenizer(include_softmasked_tokens=True).to_config())
)
sorted(p.name for p in model_dir.iterdir())


In [ ]:
random.seed(0)
fasta = work / "contig.fa"
sequence = "".join(random.choice("ACGTacgt") for _ in range(12_000))
fasta.write_text(
    ">contig_A\n"
    + "\n".join(sequence[i : i + 60] for i in range(0, len(sequence), 60))
    + "\n"
)
print(fasta.read_text()[:180])


In [ ]:
from nucleotide_transformer_v3.gene_annotation.annotate_genome import run
from nucleotide_transformer_v3.gene_annotation.config import (
    ProductionAnnotationConfig,
)

output = work / "contig.gff3"
run(ProductionAnnotationConfig(
    fasta_paths=[str(fasta)],
    weights_dir=str(model_dir),
    output_path=str(output),
    sequence_length=4096,      # required without a model_cfg
    use_bf16_compute=False,    # this toy model is fp32
    decoder="argmax",          # skip the Viterbi decode for speed
    min_cds_length=0,          # keep the toy genes
    apply_biological_filters=False,
))


### What it wrote

The `##ntv3-*` comments record the weights, geometry, decoder and filters,
so anyone receiving the file can tell how it was produced.


In [ ]:
print(output.read_text()[:1200])


## Reading the output

- **`.gff3` / `.gff` gives GFF3, `.gtf` gives GTF**, chosen by the extension;
  add `.gz` to gzip. GFF3 rows are `gene`, `CDS` and `intron` only, with
  no `mRNA` row; GTF adds one transcript per gene (`<gene_id>.t1`), since
  the decoder emits one state path per strand.
- **CDS rows include the terminal stop codon**, which is AUGUSTUS' and
  Tiberius' convention.
- **Two filters are on by default** and both change the file: genes whose
  total CDS is under `min_cds_length` (201) are dropped, as are genes whose
  spliced CDS has an internal in-frame stop. A metric computed on filtered
  output is not comparable to one computed on unfiltered output.
- **`--resume`** restarts a killed genome-scale job where it stopped.

See [the documentation](../../docs/ntv3_genome_annotation.md) for hardware,
inputs and the full option list.
